# Scikit-learn — Part 1 (ML Fundamentals & Regression)

In [ ]:
# Day 27 — Scikit-learn Part 1
# Scikit-learn (sklearn) is Python's MOST IMPORTANT ML library
# It provides tools for the ENTIRE machine learning workflow:
#   preprocessing → training → evaluation → prediction
#
# Real-world analogy:
# Think of sklearn as a FACTORY for building ML models.
# You feed in raw data, the factory cleans it, trains a model,
# and gives you a working predictor that can make predictions on new data.
#
# Today we'll cover:
#   1. What is Machine Learning (ML) — the big picture
#   2. Supervised vs Unsupervised learning
#   3. The sklearn workflow (fit → predict → evaluate)
#   4. Train/Test split — why and how
#   5. Linear Regression — predicting numbers
#   6. Model evaluation metrics (MAE, MSE, R²)
#   7. Feature scaling (StandardScaler)
#
# Install: pip install scikit-learn  (already in Anaconda)

### What is Machine Learning?

In [ ]:
# Machine Learning = teaching a computer to learn PATTERNS from DATA
# instead of programming explicit rules
#
# Traditional programming:
#   Rules + Data → Output
#   Example: if cpu > 80: alert('critical')
#
# Machine Learning:
#   Data + Output → Rules (the model LEARNS the rules)
#   Example: Give it 10,000 server readings + their labels (ok/critical)
#            The model figures out WHAT makes a server critical
#
# Two main types:
#
# 1. SUPERVISED Learning — you give the model LABELLED data
#    (input → correct answer) and it learns the mapping
#    - Regression: predict a NUMBER (price, temperature, latency)
#    - Classification: predict a CATEGORY (spam/not spam, cat/dog)
#
# 2. UNSUPERVISED Learning — you give the model UNLABELLED data
#    and it finds PATTERNS on its own
#    - Clustering: group similar items (customer segments)
#    - Dimensionality reduction: PCA (Day 16!)
#
# Today we focus on SUPERVISED LEARNING (most common in industry)

### The Scikit-learn Workflow

In [ ]:
# Every sklearn model follows the SAME 3-step pattern:
#
# Step 1: CREATE the model
#   model = SomeModel()         # e.g., LinearRegression()
#
# Step 2: TRAIN the model (fit)
#   model.fit(X_train, y_train) # X = features, y = target
#
# Step 3: PREDICT with the model
#   predictions = model.predict(X_test)
#
# That's it! Every model in sklearn follows this pattern:
# LinearRegression, DecisionTree, RandomForest, SVM, KNN, etc.
# Learn one, you've learned them ALL.
#
# Key terms:
# X (features)  = the INPUT data (columns used to make predictions)
# y (target)    = the OUTPUT we want to predict
# fit()         = TRAIN the model on data
# predict()     = USE the trained model on new data
# score()       = EVALUATE how good the model is

### Train/Test Split

In [ ]:
# WHY split data into train and test?
#
# Imagine studying for an exam:
# - If you practice with the SAME questions that appear on the exam,
#   you'll score 100% — but you didn't actually LEARN anything.
#   You just memorised the answers. This is called OVERFITTING.
#
# - The REAL test is: can you answer UNSEEN questions?
#
# Same with ML:
# - Train the model on 80% of data (training set)
# - Test it on 20% of data it has NEVER seen (test set)
# - If it performs well on test data → it actually learned!

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Create sample data: predict server response time from CPU and memory
np.random.seed(42)
n = 200

cpu = np.random.randint(10, 100, n)
memory = np.random.randint(20, 95, n)
# Response time roughly depends on CPU and memory (with some noise)
response_time = 50 + 1.5 * cpu + 0.8 * memory + np.random.normal(0, 15, n)

df = pd.DataFrame({'cpu': cpu, 'memory': memory, 'response_time': response_time})
print(df.head())
print(f'Total samples: {len(df)}')

In [ ]:
# Split into features (X) and target (y)

X = df[['cpu', 'memory']]        # features (what we use to predict)
y = df['response_time']          # target (what we want to predict)

print('X shape:', X.shape)       # (200, 2) -> 200 samples, 2 features
print('y shape:', y.shape)       # (200,)   -> 200 target values

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    random_state=42      # for reproducibility (same split every time)
)

print(f'\nTraining set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

# The model will LEARN from X_train/y_train
# and be EVALUATED on X_test/y_test (data it has never seen)

### Linear Regression — Predicting Numbers

In [ ]:
# Linear Regression = draw the best-fit LINE through the data
# It finds the formula: y = w1*x1 + w2*x2 + ... + b
# where w = weights (how much each feature matters)
#       b = bias (base value)
#
# Our model will learn:
# response_time = w1 * cpu + w2 * memory + b

from sklearn.linear_model import LinearRegression

# Step 1: Create the model
model = LinearRegression()

# Step 2: Train (fit) the model
model.fit(X_train, y_train)

# Step 3: Look at what the model learned
print('Weights (coefficients):', model.coef_)
print('Bias (intercept):', model.intercept_)

# This means:
# response_time ≈ coef[0]*cpu + coef[1]*memory + intercept
# The model learned HOW MUCH each feature affects response time!

In [ ]:
# Step 3: Make predictions on TEST data

y_pred = model.predict(X_test)

# Compare predictions vs actual values
import pandas as pd

comparison = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10].round(1)
})
print(comparison)

# Make a prediction for a NEW server
import numpy as np
new_server = np.array([[85, 70]])    # cpu=85, memory=70
predicted_rt = model.predict(new_server)
print(f'\nPredicted response time for CPU=85, Memory=70: {predicted_rt[0]:.1f}ms')

### Model Evaluation — How Good Is Our Model?

In [ ]:
# We need METRICS to measure how good our model's predictions are
#
# For REGRESSION (predicting numbers), common metrics:
#
# 1. MAE (Mean Absolute Error) — average absolute difference
#    "On average, our prediction is off by X units"
#    Lower is better. Easy to interpret.
#
# 2. MSE (Mean Squared Error) — average squared difference
#    Penalises LARGE errors more heavily than MAE
#    Lower is better.
#
# 3. RMSE (Root Mean Squared Error) — square root of MSE
#    Same units as the target. Very common in practice.
#    Lower is better.
#
# 4. R² Score — how much variance the model explains
#    1.0 = perfect, 0.0 = useless (just predicting the mean)
#    Higher is better.

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'MAE:  {mae:.2f} ms')     # average error in ms
print(f'MSE:  {mse:.2f}')        # squared error
print(f'RMSE: {rmse:.2f} ms')    # root of MSE (same units as target)
print(f'R²:   {r2:.4f}')         # 1.0 = perfect fit

# You can also use model.score() for R²
print(f'R² (via score): {model.score(X_test, y_test):.4f}')

In [ ]:
# Visualise: Actual vs Predicted

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='#3498db', edgecolors='white')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Perfect prediction')    # diagonal line

plt.title('Actual vs Predicted Response Time', fontsize=14, fontweight='bold')
plt.xlabel('Actual (ms)', fontsize=12)
plt.ylabel('Predicted (ms)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Points close to the red line = good predictions
# Points far from the red line = bad predictions

### Feature Scaling — StandardScaler

In [ ]:
# WHY scale features?
#
# Imagine two features:
#   - age: ranges from 1 to 100
#   - salary: ranges from 20,000 to 500,000
#
# Salary has MUCH larger numbers → the model might think salary is
# more important just because it's bigger (not because it matters more)
#
# Scaling puts ALL features on the SAME scale
# StandardScaler: transforms to mean=0, std=1 for each feature

from sklearn.preprocessing import StandardScaler
import numpy as np

# Example: features with very different ranges
X_example = np.array([[25, 50000],
                      [30, 60000],
                      [35, 80000],
                      [40, 45000]])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_example)

print('Before scaling:')
print(X_example)
print('\nAfter scaling:')
print(X_scaled.round(2))

# Now both columns have similar ranges (roughly -2 to +2)
# IMPORTANT: fit_transform on TRAIN data, only transform on TEST data
# scaler.fit_transform(X_train)  -> learn the mean/std AND transform
# scaler.transform(X_test)       -> use the SAME mean/std to transform

In [ ]:
# Complete workflow with scaling

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np
import pandas as pd

# 1. Prepare data
np.random.seed(42)
n = 200
cpu = np.random.randint(10, 100, n)
memory = np.random.randint(20, 95, n)
response_time = 50 + 1.5 * cpu + 0.8 * memory + np.random.normal(0, 15, n)

X = np.column_stack([cpu, memory])
y = response_time

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)     # fit + transform on TRAIN
X_test_scaled = scaler.transform(X_test)            # only transform on TEST

# 4. Train
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# 5. Predict & Evaluate
y_pred = model.predict(X_test_scaled)
print(f'R²:  {r2_score(y_test, y_pred):.4f}')
print(f'MAE: {mean_absolute_error(y_test, y_pred):.2f} ms')

# This is the STANDARD ML workflow you'll use in every project

In [ ]:
# Practice Questions:

# 1. Load the built-in Boston-style housing data (or create synthetic data)
#    with features: area_sqft, bedrooms, age_years, distance_to_city
#    Target: price
#    Train a LinearRegression and report R² and MAE

# 2. Try training WITHOUT scaling and WITH scaling
#    Compare the R² scores — is there a difference for Linear Regression?

# 3. Create a scatter plot of actual vs predicted prices
#    Add the perfect-prediction diagonal line

# 4. Print the model coefficients and interpret them:
#    "For every 1 unit increase in X, the price changes by Y"

# 5. What happens if you use test_size=0.1 vs test_size=0.5?
#    Run both and compare R² scores. Why might they differ?

### DevOps / AI-MLOps Practice

In [ ]:
# sklearn is the foundation of every ML pipeline

# Exercise 1: API Latency Predictor
# Create synthetic data:
#   features: concurrent_users, payload_size_kb, hour_of_day, is_weekend
#   target: response_time_ms
# Train a LinearRegression model
# Evaluate with MAE, RMSE, R²
# Predict latency for: 500 users, 2KB payload, 14:00, weekday

# Exercise 2: Cloud Cost Predictor
# features: cpu_hours, storage_gb, network_gb, num_instances
# target: monthly_cost_usd
# a) Generate synthetic data (200 samples)
# b) Split 80/20, scale with StandardScaler
# c) Train LinearRegression
# d) Print which feature has the HIGHEST coefficient (most expensive)
# e) Predict cost for: 1000 cpu_hrs, 500GB storage, 200GB network, 5 instances

# Exercise 3: The Full Pipeline
# Write a FUNCTION that takes X, y and:
# a) Splits into train/test (80/20)
# b) Scales features
# c) Trains a LinearRegression
# d) Evaluates on test set
# e) Returns a dict: {'model': model, 'scaler': scaler, 'r2': r2, 'mae': mae}
# This is how you'd structure a reusable ML training function